# Simulate a controller

Build a universal co-Büchi automaton (UCB), solve its bounded safety game, and
inspect the outputs that keep each successor in the winning region. You can
also try larger bounds after choosing an output outside that region.

The specification is `G F req -> G F grant`. Pass its negation to `create_twa`,
as explained in the [Python guide](../docs/python.md#solve-a-bounded-safety-game).


In [ ]:
import itertools

import acacia_boomslang as ab
import spot

SPEC = "!((G (F (req))) -> (G (F (grant))))"
INPUTS  = ["req"]
OUTPUTS = ["grant"]

INITIAL_K = 2
K_STEP    = 3
MAX_K     = 30

## Build and inspect the automaton

`create_twa` returns a `Game` containing the automaton and its input/output
propositions. Use `get_aut_hoa` to export it and Spot to display it.


In [ ]:
game = ab.create_twa(SPEC, INPUTS, OUTPUTS)
print(f"UCB: {ab.num_states(game)} states, initial = {ab.initial_state_number(game)}")

aut = spot.automaton(ab.get_aut_hoa(game))
aut

## Solve the safety game

Preprocess the automaton, set its counter representation, and solve with bound
`k`. A winning region proves realizability. Finding none at this bound does
not prove the LTL specification unrealizable.


In [ ]:
def solve(game, k):
    ab.preprocess_aut_standard(game, k_max=k)
    ab.set_bool_thresh_no_bool_states(game, k_max=k)
    return ab.solve_acacia_safety_game(game, k_max=k, k_min=min(2, k), k_inc=K_STEP)

k = INITIAL_K
result = solve(game, k)
assert result.is_real(), "no winning region at INITIAL_K; try a larger bound"

winreg = result.get_winning_region()
print(f"Realizable at k={k}; winning region has {len(winreg)} antichain heads.")
for i, v in enumerate(winreg):
    print(f"  head {i}:", list(v))

## Classify output choices

Fix the inputs and compute the successor for each output assignment. Outputs
that keep the successor in the winning region form the controller's choices
at this state.


In [ ]:
def all_assignments(aps):
    for bits in itertools.product([False, True], repeat=len(aps)):
        t = [a for a, b in zip(aps, bits) if b]
        f = [a for a, b in zip(aps, bits) if not b]
        yield t, f

def label(t, f):
    return " & ".join([*t, *(f"!{a}" for a in f)]) or "tt"

def classify_outputs(game, v, true_inputs, false_inputs, winreg, k_cap):
    winning, losing = [], []
    for t_out, f_out in all_assignments(OUTPUTS):
        s = ab.successor(
            game, v,
            ab.StringVector(true_inputs + t_out),
            ab.StringVector(false_inputs + f_out),
            k_cap)
        (winning if winreg.contains(s) else losing).append((t_out, f_out, s))
    return winning, losing

v = ab.get_initial_state(game)
print("state vector:", [v[i] for i in range(len(v))])

winning, losing = classify_outputs(game, v, ["req"], [], winreg, k)
print("under input req=T, grant should be one of the winning outputs:")
for t, f, s in winning:
    print(f"  W  {label(t, f):>12}  -> {[s[i] for i in range(len(s))]}")
for t, f, s in losing:
    print(f"  L  {label(t, f):>12}  -> {[s[i] for i in range(len(s))]}")

## Take one step

Pick the first winning output under `req=True` and update the state vector.


In [ ]:
t_out, f_out, succ = winning[0]
print(f"pick output {label(t_out, f_out)} -> successor {[succ[i] for i in range(len(succ))]}")
v = succ

## Try a larger bound

An output outside the current winning region may lead to a state covered by a
region at a larger bound. The helper below tries bounds up to `MAX_K` and
returns a new game and region if it finds one. Failure to recover within that
limit does not prove unrealizability.


In [ ]:
def try_grow_k(game, escaped_vec, current_k):
    new_k = current_k
    while new_k + K_STEP <= MAX_K:
        new_k += K_STEP
        fresh_game = ab.create_twa(SPEC, INPUTS, OUTPUTS)
        fresh_result = solve(fresh_game, new_k)
        if not fresh_result.is_real():
            continue
        fresh_w = fresh_result.get_winning_region()
        recast = ab.make_vector(fresh_game, ab.IntVector([escaped_vec[i] for i in range(len(escaped_vec))]))
        if fresh_w.contains(recast):
            return fresh_game, fresh_w, recast, new_k
    return None

# Force a "losing" step by picking a (hypothetically) bad output if any losing
# choices exist at this state.
winning, losing = classify_outputs(game, v, [], ["req"], winreg, k)
print("under input !req, winning outputs:")
for t, f, s in winning:
    print("  W", label(t, f), "->", [s[i] for i in range(len(s))])
print("losing outputs:")
for t, f, s in losing:
    print("  L", label(t, f), "->", [s[i] for i in range(len(s))])

if losing:
    t, f, s = losing[0]
    print(f"\npicking losing output {label(t, f)} -> {[s[i] for i in range(len(s))]}")
    grown = try_grow_k(game, s, k)
    if grown:
        game, winreg, v, k = grown
        print(f"recovered at k={k}; winning region has {len(winreg)} heads.")
    else:
        print("no larger k up to MAX_K recovers this state.")
else:
    print("no losing outputs at this state under the chosen input.")

## Run the simulation script

From the repository root, with the module path set as in the
[Python guide](../docs/python.md), run:

```sh
python3 python_examples/example_simulate.py auto
```

Omit `auto` to choose inputs and outputs interactively.
